# Mock 피드백 생성 후 재생성 테스트

In [6]:
# 환경 변수
from dotenv import load_dotenv
load_dotenv()

# DB 함수들
from content_pipeline.scenario.scenario_database import (
    get_db_connection,
    load_company_data,
    load_meme_data
)

# 노드 함수
from content_pipeline.scenario.scenario_nodes import regenerate_scenario_node

# 스키마
from content_pipeline.scenario.scenario_schemas import Scenario

# State 타입 (타입 힌트용, 선택)
from content_pipeline.scenario.scenario_state import ScenarioState

# 프롬프트 템플릿
from content_pipeline.scenario.prompts.regenerate_scenario_templates import REGENERATE_SCENARIO_TEMPLATE_V2
from content_pipeline.scenario.prompts.meme_example_templates import MEME_EXAMPLE_TEMPLATE_V1

# DB 관련
import psycopg2
from psycopg2.extras import RealDictCursor

# 기타
from typing import Dict, Any

In [7]:
# DB에서 실제 저장된 시나리오 가져오기
conn = get_db_connection()
cursor = conn.cursor(cursor_factory=RealDictCursor)

cursor.execute("""
    SELECT * FROM scenario_scripts 
    WHERE script_id = %s
""", (18,))
saved_scenario = cursor.fetchone()

# Pydantic 객체로 변환
scenario = Scenario(**saved_scenario['scenes'])

In [3]:
scenario

Scenario(hook=Scene(scene_number=1, duration_seconds=4.0, dialogue='"난 오늘 네가 킹받으면 좋겠어!"', emotion='playful', action='캐릭터가 스마트워치를 손목에 차고 귀여운 춤을 춘다.', visual_description="밝은 배경에서 캐릭터가 스마트워치를 강조하며 춤을 춘다. 화면에 '건강 관리의 새로운 기준'이라는 텍스트가 나타난다."), body=[Scene(scene_number=2, duration_seconds=5.0, dialogue='"이 스마트워치로 건강 관리도 킹받게!"', emotion='enthusiastic', action='캐릭터가 스마트워치의 다양한 기능을 보여주며 손목을 흔든다.', visual_description='캐릭터가 스마트워치의 심박수 측정, 운동 기록 기능을 시연하며 화면에 기능 설명이 나타난다.'), Scene(scene_number=3, duration_seconds=4.0, dialogue='"건강 관리가 이렇게 재밌을 줄이야!"', emotion='joyful', action='캐릭터가 스마트워치로 운동 목표를 달성하고 기뻐한다.', visual_description='캐릭터가 운동 목표 달성 알림을 받고 환하게 웃으며 스마트워치를 바라본다.')], close=Scene(scene_number=4, duration_seconds=4.0, dialogue='"테스트기업의 신제품 스마트워치, 지금 검색하세요!"', emotion='motivating', action='캐릭터가 손목을 가리키며 시청자에게 손짓한다.', visual_description="캐릭터가 스마트워치를 강조하며 화면에 '지금 검색하세요!'라는 텍스트가 나타난다."), title='스마트워치로 킹받게!', description='밈과 함께하는 스마트워치의 건강 관리 이야기', hashtags=['킹받으면좋겠어', '스마트워치', '건강관리', '테스트기업'], tota

In [8]:
# 실제 회사/밈 데이터도 DB에서
company_data = load_company_data(saved_scenario['ad_id'])
meme_data = load_meme_data(
    saved_scenario['meme_id'],
    MEME_EXAMPLE_TEMPLATE_V1
)

In [5]:
company_data, meme_data

({'company_name': '테스트기업',
  'item_name': '신제품 스마트워치',
  'item_category': '전자제품',
  'item_keymessage': '건강 관리의 새로운 기준',
  'character_mood': '밝고 활기찬'},
 {'meme_name': '난 오늘 네가 킹받으면 좋겠어',
  'definition': '"난 오늘 네가 킹받으면 좋겠어는 인터넷에서 유행하는 밈이자 노래로, 중독성 있는 멜로디와 귀여운 캐릭터 영상으로 주목받고 있다. 이 밈은 \'행복한 피자빵\'이 제작하였으며, 카사네 테토의 목소리를 사용하였다. 밈은 주로 틱톡과 유튜브 쇼츠에서 인기를 끌며, \'킹받는다\'라는 표현과 함께 다양한 변형과 패러디가 제작되었다. 사람들은 이 밈을 통해 유머와 재미를 공유하며, 특정 상황에서 상대방을 놀려주는 재미로 활용한다. \'킹받는다\'는 \'화가 난다\'는 의미의 신조어로, 감정을 재치 있게 표현하는 데 활용된다."',
  'key_phrase': '난 오늘 네가 킹받으면 좋겠어',
  'usage_examples': "\n## 활용 예시 1\n- 사용 맥락: 블로그에서 친한 친구에게 장난스럽게 공유\n- 사용 예시: '너 오늘 킹 받으라고 보내거나 블로그 제목으로 작성하면 좋을 것 같아요!'\n- 예시 분류: good\n- 예시 분류가 bad일 경우 사유: 해당 없음\n- 예시 톤: playful\n\n\n## 활용 예시 2\n- 사용 맥락: 틱톡에서 다양한 상황에서의 킹받음 공유\n- 사용 예시: '오늘 너가 킹받는 모습이 보고 싶어! 다양한 상황에서의 킹받음 공유!'\n- 예시 분류: good\n- 예시 분류가 bad일 경우 사유: 해당 없음\n- 예시 톤: playful\n"})

In [9]:
# 여기만 조작 - 일부러 낮은 점수
mock_review = {
    "approved": False,  # 여기만 수정
    "total_score": 186.0,  # 그대로 유지
    "feedback_type": "llm_review",
    "scene_reviews": [
        {
            "scene_number": 1,
            "feedback_items": [
                {
                    "category": "message_delivery",
                    "description": "밈과 상품의 연결점이 명확하나, '건강 관리의 새로운 기준' 메시지가 더 강조될 필요 있음."
                }
            ],
            "meme_usage_score": 9.0,
            "brand_safety_score": 9.0,
            "completeness_score": 9.0,
            "message_delivery_score": 8.0,
            "character_consistency_score": 10.0
        },
        {
            "scene_number": 2,
            "feedback_items": [
                {
                    "category": "meme_usage",
                    "description": "밈의 참신함이 다소 부족. '킹받게'라는 표현을 더 창의적으로 활용할 필요 있음."
                }
            ],
            "meme_usage_score": 8.0,
            "brand_safety_score": 9.0,
            "completeness_score": 9.0,
            "message_delivery_score": 9.0,
            "character_consistency_score": 10.0
        },
        # 씬 3, 4는 그대로
    ],
    "scene_feedback": {
        "1": "밈과 상품의 연결점이 명확하나, '건강 관리의 새로운 기준' 메시지가 더 강조될 필요 있음.",
        "2": "밈의 참신함이 다소 부족. '킹받게'라는 표현을 더 창의적으로 활용할 필요 있음."
    },
    "overall_comment": "일부 씬에서 메시지 강조와 밈 활용의 참신함을 보완하면 더욱 완성도 높은 광고가 될 것입니다."
}

In [10]:
test_state = {
    "ad_id": saved_scenario['ad_id'],
    "meme_id": saved_scenario['meme_id'],
    "scenario": scenario,  # 실제 데이터
    "company_data": company_data,  # 실제 데이터
    "meme_data": meme_data,  # 실제 데이터
    "review_result": mock_review,  # Mock
    "retry_count": 0,
    "generation_type": "initial",
    "regenerate_template": REGENERATE_SCENARIO_TEMPLATE_V2,
    "status": "reviewed"
}

# 실제 LLM 호출 발생
result = regenerate_scenario_node(test_state)

In [11]:
# DB 저장 (save_scenario_to_db 직접 호출)
from content_pipeline.scenario.scenario_database import save_scenario_to_db

script_id = save_scenario_to_db(
    scenario=result['scenario'].model_dump(),
    ad_id=test_state['ad_id'],
    meme_id=test_state['meme_id'],
    generation_type="regeneration_test_2",  # 테스트 표시
    used_templates=None,  
    model="gpt-4o"
)

print(f"저장 완료: script_id={script_id}")

저장 완료: script_id=32


In [13]:
# 실제로 변경되었는지
print("=== 변경 전 ===")
print(f"Title: {scenario.title}")
print(f"Hook: {scenario.hook.dialogue}")

print("\n=== 변경 후 ===")
print(f"Title: {result['scenario'].title}")
print(f"Hook: {result['scenario'].hook.dialogue}")

# 피드백이 반영되었는지 육안 확인

=== 변경 전 ===
Title: 스마트워치로 킹받게!
Hook: "난 오늘 네가 킹받으면 좋겠어!"

=== 변경 후 ===
Title: 스마트워치로 킹받게!
Hook: "난 오늘 네가 킹받으면 좋겠어! 건강 관리의 새로운 기준, 준비됐어?"


In [14]:
result

{'ad_id': 10,
 'meme_id': 159,
 'scenario': Scenario(hook=Scene(scene_number=1, duration_seconds=4.0, dialogue='"난 오늘 네가 킹받으면 좋겠어! 건강 관리의 새로운 기준, 준비됐어?"', emotion='playful', action='캐릭터가 스마트워치를 손목에 차고 귀여운 춤을 추며 손목을 가리킨다.', visual_description="밝은 배경에서 캐릭터가 스마트워치를 강조하며 춤을 춘다. 화면에 '건강 관리의 새로운 기준, 준비됐어?'라는 텍스트가 나타난다."), body=[Scene(scene_number=2, duration_seconds=5.0, dialogue='"이 스마트워치로 건강 관리도 킹받게! 매일매일 새로운 도전!"', emotion='enthusiastic', action='캐릭터가 스마트워치의 다양한 기능을 보여주며 손목을 흔들고, 화면에 도전 과제가 나타난다.', visual_description="캐릭터가 스마트워치의 심박수 측정, 운동 기록 기능을 시연하며 화면에 '매일매일 새로운 도전!'이라는 텍스트가 나타난다."), Scene(scene_number=3, duration_seconds=4.0, dialogue='"건강 관리가 이렇게 재밌을 줄이야! 킹받는 성취감!"', emotion='joyful', action='캐릭터가 스마트워치로 운동 목표를 달성하고 기뻐하며 손목을 하이파이브한다.', visual_description="캐릭터가 운동 목표 달성 알림을 받고 환하게 웃으며 스마트워치를 바라본다. 화면에 '킹받는 성취감!'이라는 텍스트가 나타난다.")], close=Scene(scene_number=4, duration_seconds=4.0, dialogue='"테스트기업의 신제품 스마트워치, 지금 검색하세요! 킹받는 변화가 시작됩니다!"', emotion='motivating', action='캐릭터가 손목을 가리키며 시청자에게

In [15]:
scenario.model_dump()

{'hook': {'scene_number': 1,
  'duration_seconds': 4.0,
  'dialogue': '"난 오늘 네가 킹받으면 좋겠어!"',
  'emotion': 'playful',
  'action': '캐릭터가 스마트워치를 손목에 차고 귀여운 춤을 춘다.',
  'visual_description': "밝은 배경에서 캐릭터가 스마트워치를 강조하며 춤을 춘다. 화면에 '건강 관리의 새로운 기준'이라는 텍스트가 나타난다."},
 'body': [{'scene_number': 2,
   'duration_seconds': 5.0,
   'dialogue': '"이 스마트워치로 건강 관리도 킹받게!"',
   'emotion': 'enthusiastic',
   'action': '캐릭터가 스마트워치의 다양한 기능을 보여주며 손목을 흔든다.',
   'visual_description': '캐릭터가 스마트워치의 심박수 측정, 운동 기록 기능을 시연하며 화면에 기능 설명이 나타난다.'},
  {'scene_number': 3,
   'duration_seconds': 4.0,
   'dialogue': '"건강 관리가 이렇게 재밌을 줄이야!"',
   'emotion': 'joyful',
   'action': '캐릭터가 스마트워치로 운동 목표를 달성하고 기뻐한다.',
   'visual_description': '캐릭터가 운동 목표 달성 알림을 받고 환하게 웃으며 스마트워치를 바라본다.'}],
 'close': {'scene_number': 4,
  'duration_seconds': 4.0,
  'dialogue': '"테스트기업의 신제품 스마트워치, 지금 검색하세요!"',
  'emotion': 'motivating',
  'action': '캐릭터가 손목을 가리키며 시청자에게 손짓한다.',
  'visual_description': "캐릭터가 스마트워치를 강조하며 화면에 '지금 검색하세요!'라는 텍스트가 나타난다."},
 'titl

In [16]:
result["scenario"].model_dump()

{'hook': {'scene_number': 1,
  'duration_seconds': 4.0,
  'dialogue': '"난 오늘 네가 킹받으면 좋겠어! 건강 관리의 새로운 기준, 준비됐어?"',
  'emotion': 'playful',
  'action': '캐릭터가 스마트워치를 손목에 차고 귀여운 춤을 추며 손목을 가리킨다.',
  'visual_description': "밝은 배경에서 캐릭터가 스마트워치를 강조하며 춤을 춘다. 화면에 '건강 관리의 새로운 기준, 준비됐어?'라는 텍스트가 나타난다."},
 'body': [{'scene_number': 2,
   'duration_seconds': 5.0,
   'dialogue': '"이 스마트워치로 건강 관리도 킹받게! 매일매일 새로운 도전!"',
   'emotion': 'enthusiastic',
   'action': '캐릭터가 스마트워치의 다양한 기능을 보여주며 손목을 흔들고, 화면에 도전 과제가 나타난다.',
   'visual_description': "캐릭터가 스마트워치의 심박수 측정, 운동 기록 기능을 시연하며 화면에 '매일매일 새로운 도전!'이라는 텍스트가 나타난다."},
  {'scene_number': 3,
   'duration_seconds': 4.0,
   'dialogue': '"건강 관리가 이렇게 재밌을 줄이야! 킹받는 성취감!"',
   'emotion': 'joyful',
   'action': '캐릭터가 스마트워치로 운동 목표를 달성하고 기뻐하며 손목을 하이파이브한다.',
   'visual_description': "캐릭터가 운동 목표 달성 알림을 받고 환하게 웃으며 스마트워치를 바라본다. 화면에 '킹받는 성취감!'이라는 텍스트가 나타난다."}],
 'close': {'scene_number': 4,
  'duration_seconds': 4.0,
  'dialogue': '"테스트기업의 신제품 스마트워치, 지금 검색하세요! 킹받는 변화가 시작됩니다!"',
 

In [17]:
# 열린 DB 연결 있으면 닫기
try:
    if 'conn' in locals() and conn:
        conn.close()
        print("DB 연결 종료")
    if 'cursor' in locals() and cursor:
        cursor.close()
except:
    pass


DB 연결 종료


In [14]:
# 1. 변수 존재 여부
print("conn 존재:", 'conn' in locals())
print("cursor 존재:", 'cursor' in locals())

# 2. 연결 상태 확인
if 'conn' in locals():
    print("conn.closed:", conn.closed)  # 0=열림, 1=닫힘

conn 존재: True
cursor 존재: True
conn.closed: 1
